# BusDemand AI – Exploratory Data Analysis (Stage 6)
## Synthetic Trip-Level Passenger Demand Analysis

This notebook provides an in-depth exploratory analysis of the **87,680 synthetic trip records** generated in Stage 6.
- **Dataset:** `data/processed/trip_demand.csv`
- **Horizon:** 2024-01-01 to 2026-12-31 (1,096 days)
- **Network:** 20 directed intercity routes across 8 South Indian hubs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Load trip demand dataset
df = pd.read_csv('../data/processed/trip_demand.csv')
print(f'Loaded {len(df):,} trips across {len(df.columns)} columns.')
df.head(3)

### 1. Overall Passenger Demand Distribution
Physical coach capacity is standardized to **45 seats**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram / KDE
axes[0].hist(df['passenger_demand'], bins=50, color='#1f77b4', edgecolor='black', alpha=0.7)
axes[0].axvline(45, color='red', linestyle='--', linewidth=2, label='Bus Capacity (45 seats)')
axes[0].axvline(df['passenger_demand'].mean(), color='green', linestyle='-', linewidth=2, label=f'Mean ({df["passenger_demand"].mean():.1f})')
axes[0].set_title('Passenger Demand Distribution')
axes[0].set_xlabel('Passenger Demand')
axes[0].set_ylabel('Trip Frequency')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['passenger_demand'], vert=False, patch_artist=True, boxprops=dict(facecolor='#aec7e8'))
axes[1].axvline(45, color='red', linestyle='--', linewidth=2, label='Bus Capacity')
axes[1].set_title('Passenger Demand Boxplot')
axes[1].set_xlabel('Passenger Demand')
axes[1].set_yticks([])
axes[1].legend()

plt.tight_layout()
plt.show()

print(df['passenger_demand'].describe())

### 2. Demand by Departure Slot

In [ ]:
slot_order = ['DEP_01', 'DEP_02', 'DEP_03', 'DEP_04']
slot_labels = ['Early Morning\n(06:00)', 'Midday\n(11:30)', 'Evening\n(17:00)', 'Night Sleeper\n(22:30)']

plt.figure(figsize=(10, 5))
means = df.groupby('departure_slot')['passenger_demand'].mean().reindex(slot_order)
bars = plt.bar(slot_labels, means, color=['#2ca02c', '#7f7f7f', '#ff7f0e', '#1f77b4'], edgecolor='black')
plt.axhline(45, color='red', linestyle='--', label='Bus Capacity (45)')
plt.title('Mean Passenger Demand by Departure Slot')
plt.ylabel('Mean Demand (seats requested)')
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 0.5, f'{h:.1f}', ha='center', va='bottom', fontweight='bold')
plt.legend()
plt.show()

### 3. Weekday vs Weekend & Day-of-Week Patterns

In [ ]:
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_means = df.groupby('day_of_week')['passenger_demand'].mean().reindex(range(7))

plt.figure(figsize=(10, 5))
plt.plot(dow_names, dow_means, marker='o', linewidth=2.5, color='#d62728', markersize=8)
plt.axhline(45, color='red', linestyle='--', alpha=0.5, label='Capacity (45)')
plt.title('Mean Demand Across Days of the Week')
plt.ylabel('Mean Passenger Demand')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

### 4. Mean Demand by Route (20 Directed Corridors)

In [ ]:
route_means = df.groupby('route_id')['passenger_demand'].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
bars = plt.barh(route_means.index, route_means.values, color='#1f77b4', edgecolor='black')
plt.axvline(45, color='red', linestyle='--', linewidth=2, label='Bus Capacity (45)')
plt.title('Mean Passenger Demand across 20 Directed Routes')
plt.xlabel('Mean Passenger Demand')
plt.legend()
plt.tight_layout()
plt.show()

### 5. Capacity Exceeded & Operational Load Factor

In [ ]:
exceeded_by_route = df.groupby('route_id')['capacity_exceeded'].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(10, 6))
exceeded_by_route.plot(kind='bar', color='#e377c2', edgecolor='black')
plt.title('Percentage of Trips Exceeding Bus Capacity (> 45 seats)')
plt.ylabel('% Trips Exceeded')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()